# F02 — CELESTIAN : La Montre
> *"Purifiez la voix comme on affûte une lame. Chaque fréquence est une décision."*
> — Ordre de la Rose Sacrée, Adepta Sororitas

```
╔══════════════════════════════════════════════════════════════╗
║   FRÉGATE F02 — CELESTIAN                                   ║
║   Rôle    : Purification DSP + Presets Personnages          ║
║   IN      : voix_brute.wav  (sortie F01_DOMINION)           ║
║   OUT     : voix_purifiee.wav                               ║
║   Stack   : Spotify Pedalboard · importlib · Gradio         ║
╚══════════════════════════════════════════════════════════════╝
```

> **Note SETUP DRIVE** : La structure Google Drive a été créée par la **Cellule 0 de F01_DOMINION**.
> Ne pas réexécuter ici.

---

## Ordre des Cellules

| # | Cellule | Run | Description |
|---|---------|-----|-------------|
| 1 | INIT | Chaque session | Monter Drive, cloner SANCTORUM, définir chemins |
| 2 | INSTALLATION | Chaque session | Installer Pedalboard, pydub, soundfile, gradio |
| 3 | INTERFACE | Chaque session | Interface Gradio — La Montre |
| 4 | SR_CUSTOS | Après purification | Check-in CMS flotte |

---
## CELLULE 1 — INIT

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — INIT                                       ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, json
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# ── Chemins Drive ───────────────────────────────────────────────────────
DRIVE_ROOT      = "/content/drive/MyDrive/SANCTORUM"
F01_OUT         = f"{DRIVE_ROOT}/F01_DOMINION/OUT"
F02_IN          = f"{DRIVE_ROOT}/F02_CELESTIAN/IN"
F02_OUT         = f"{DRIVE_ROOT}/F02_CELESTIAN/OUT"
F02_TRACKING    = f"{DRIVE_ROOT}/F02_CELESTIAN/TRACKING"
PRESETS_DIR     = f"{DRIVE_ROOT}/F02_CELESTIAN/CODEBASE/presets"
LIBER_DRIVE     = f"{DRIVE_ROOT}/liber_sanctorum.json"

# ── Chemins locaux (Colab) ───────────────────────────────────────────────
SANCTORUM_DIR   = "/content/SANCTORUM"
LOCAL_PRESETS   = f"{SANCTORUM_DIR}/F02_CELESTIAN/CODEBASE/presets"

# ── Cloner SANCTORUM si absent ───────────────────────────────────────────
if not os.path.exists(SANCTORUM_DIR):
    print("[INIT] Clonage SANCTORUM...")
    !git clone https://github.com/kioka8877-ux/SANCTORUM.git {SANCTORUM_DIR} -q
else:
    print("[INIT] SANCTORUM présent — pull...")
    !git -C {SANCTORUM_DIR} pull -q

if SANCTORUM_DIR not in sys.path:
    sys.path.insert(0, SANCTORUM_DIR)

# ── Garantir les dossiers Drive ──────────────────────────────────────────
for d in [F02_IN, F02_OUT, F02_TRACKING]:
    os.makedirs(d, exist_ok=True)

# ── Lire le liber & détecter l'entrée ───────────────────────────────────
fleet_status = 'unknown'
f01_output   = ''
if os.path.exists(LIBER_DRIVE):
    with open(LIBER_DRIVE) as f:
        liber = json.load(f)
    fleet_status = liber.get('fleet_status', 'unknown')
    f01_output   = liber.get('f01_dominion', {}).get('output_path', '')

# Chemin de l'entrée F02 (priorité : liber → emplacement standard F01)
F02_INPUT_AUTO = (
    f01_output if (f01_output and os.path.exists(f01_output))
    else os.path.join(F01_OUT, 'voix_brute.wav')
)

print(f"\n[INIT] fleet_status  : {fleet_status}")
print(f"[INIT] F01 output    : {f01_output or '(non défini dans le liber)'}")
print(f"[INIT] Entrée auto   : {F02_INPUT_AUTO} "
      f"({'EXISTE' if os.path.exists(F02_INPUT_AUTO) else 'ABSENT — upload manuel requis'})")
print("[INIT] Prêt.")

---
## CELLULE 2 — INSTALLATION

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — INSTALLATION                               ║
# ╚══════════════════════════════════════════════════════════╝

print("[INSTALL] Dépendances F02 CELESTIAN...")
!pip install -q pedalboard>=0.9.0
!pip install -q pydub>=0.25.1 soundfile>=0.12.0 numpy>=1.24.0
!pip install -q gradio>=4.31.0
!pip install -q pyloudnorm matplotlib

# ffmpeg pour pydub
!apt-get install -qq ffmpeg 2>/dev/null

print("[INSTALL] Terminé.")

---
## CELLULE 3 — INTERFACE GRADIO — LA MONTRE

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — INTERFACE GRADIO — LA MONTRE               ║
# ║  Purification DSP + Presets Personnages                  ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, json, shutil, importlib, hashlib, time, tempfile
from datetime import datetime, timezone

import numpy as np
import soundfile as sf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pydub import AudioSegment
from pedalboard import Pedalboard, HighpassFilter, LowpassFilter, Compressor, Reverb, Gain, Distortion
import gradio as gr

# ── Chemins (définis en cellule 1, redéfinis ici pour autonomie) ─────────
DRIVE_ROOT   = "/content/drive/MyDrive/SANCTORUM"
F01_OUT      = f"{DRIVE_ROOT}/F01_DOMINION/OUT"
F02_OUT      = f"{DRIVE_ROOT}/F02_CELESTIAN/OUT"
F02_TRACKING = f"{DRIVE_ROOT}/F02_CELESTIAN/TRACKING"
LIBER_DRIVE  = f"{DRIVE_ROOT}/liber_sanctorum.json"
SANCTORUM_DIR = "/content/SANCTORUM"
LOCAL_PRESETS = f"{SANCTORUM_DIR}/F02_CELESTIAN/CODEBASE/presets"


# ╔══════════════════════════════════════════════════════════╗
# ║  CHARGEMENT DYNAMIQUE DES PRESETS (importlib)            ║
# ╚══════════════════════════════════════════════════════════╝

def discover_presets(presets_dir: str) -> dict:
    """Charger dynamiquement tous les presets .py du dossier presets/."""
    presets = {}
    if not os.path.isdir(presets_dir):
        return presets
    if presets_dir not in sys.path:
        sys.path.insert(0, presets_dir)
    for fname in sorted(os.listdir(presets_dir)):
        if not fname.endswith('.py') or fname.startswith('_'):
            continue
        mod_name = fname[:-3]
        try:
            spec   = importlib.util.spec_from_file_location(mod_name,
                         os.path.join(presets_dir, fname))
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            except Exception as e:
            print(f"[PRESETS] Impossible de charger {fname}: {e}")
            continue
        if not callable(getattr(module, 'apply', None)):
            print(f"[PRESETS] {fname} ignoré — pas de fonction apply()")
            continue
        presets[mod_name] = module
    return presets

PRESETS = discover_presets(LOCAL_PRESETS)
print(f"[PRESETS] {len(PRESETS)} preset(s) chargé(s): {list(PRESETS.keys())}")


# ╔══════════════════════════════════════════════════════════╗
# ║  PIPELINE DSP                                            ║
# ╚══════════════════════════════════════════════════════════╝

def _normalize_lufs(audio_data: np.ndarray, sr: int, target_lufs: float = -16.0) -> np.ndarray:
    """Normalisation LUFS (pyloudnorm). Fallback RMS si indisponible."""
    try:
        import pyloudnorm as pyln
        meter = pyln.Meter(sr)
        loudness = meter.integrated_loudness(audio_data.T if audio_data.ndim > 1 else audio_data)
        if np.isfinite(loudness):
            gain_db = target_lufs - loudness
            gain_lin = 10 ** (gain_db / 20.0)
            return np.clip(audio_data * gain_lin, -1.0, 1.0)
    except Exception:
        pass
    # Fallback : peak normalization
    peak = np.max(np.abs(audio_data))
    if peak > 0:
        audio_data = audio_data / peak * 0.95
    return audio_data


def _noise_gate(audio_data: np.ndarray, threshold_db: float = -50.0) -> np.ndarray:
    """Porte de bruit simple — met à zéro les segments sous threshold_db."""
    if threshold_db <= -100:
        return audio_data
    threshold_lin = 10 ** (threshold_db / 20.0)
    mask = np.abs(audio_data) < threshold_lin
    result = audio_data.copy()
    result[mask] = 0.0
    return result


def apply_dsp_pipeline(
    input_path: str,
    preset_name: str,
    # Surcharges DSP individuelles
    highpass_hz: float      = 75.0,
    comp_threshold: float   = -15.0,
    comp_ratio: float       = 3.5,
    reverb_room: float      = 0.20,
    reverb_wet: float       = 0.08,
    presence_boost: bool    = False,
    presence_gain_db: float = 3.0,
    noise_gate_db: float    = -50.0,
    target_lufs: float      = -16.0,
    use_preset_defaults: bool = True,
) -> tuple:
    """
    Pipeline complet :
      1. Lire l'audio
      2. Appliquer le preset (ou les paramètres manuels)
      3. Porte de bruit
      4. Boost de présence (optionnel)
      5. Normalisation LUFS
    Retourne (audio_processed, sample_rate).
    """
    audio, sr = sf.read(input_path, always_2d=False)
    # Convertir en float32 mono ou stéréo
    if audio.dtype != np.float32:
        audio = audio.astype(np.float32)
    if np.abs(audio).max() > 1.0:
        audio = audio / np.abs(audio).max()

    # Mise en forme pour pedalboard (channels-first si stéréo)
    if audio.ndim == 1:
        pb_audio = audio.reshape(1, -1)
    else:
        pb_audio = audio.T  # (channels, samples)

    if use_preset_defaults and preset_name in PRESETS:
        # ── Chemin preset : convertir en AudioSegment puis appliquer ────
        module = PRESETS[preset_name]
        tmp_in = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        tmp_in.close()
        sf.write(tmp_in.name, audio, sr, subtype='PCM_16')
        seg = AudioSegment.from_wav(tmp_in.name)
        os.unlink(tmp_in.name)

        seg_out = module.apply(seg)

        tmp_out = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        tmp_out.close()
        seg_out.export(tmp_out.name, format='wav')
        processed, sr = sf.read(tmp_out.name, always_2d=False)
        os.unlink(tmp_out.name)
        if processed.dtype != np.float32:
            processed = processed.astype(np.float32)
        if processed.max() > 1.0:
            processed /= 32768.0
        pb_audio = processed.reshape(1, -1) if processed.ndim == 1 else processed.T
    else:
        # ── Chemin manuel : construire la pedalboard depuis les sliders ─
        chain = [
            HighpassFilter(cutoff_frequency_hz=highpass_hz),
            Compressor(threshold_db=comp_threshold, ratio=comp_ratio,
                       attack_ms=5.0, release_ms=100.0),
            Reverb(room_size=reverb_room, wet_level=reverb_wet,
                   dry_level=1.0 - reverb_wet, damping=0.5),
        ]
        board = Pedalboard(chain)
        pb_audio = board(pb_audio, sr)

    # Repasser en mono/stéréo numpy (samples-last)
    if pb_audio.ndim == 2:
        processed = pb_audio.T if pb_audio.shape[0] <= 2 else pb_audio
    else:
        processed = pb_audio

    # ── Boost de présence (bande 2–5kHz) ────────────────────────────────
    if presence_boost and presence_gain_db != 0:
        from scipy.signal import butter, lfilter
        b, a = butter(2, [2000/(sr/2), 5000/(sr/2)], btype='band')
        presence_band = lfilter(b, a, processed, axis=0)
        gain_lin = 10 ** (presence_gain_db / 20.0)
        processed = processed + presence_band * (gain_lin - 1.0)
        processed = np.clip(processed, -1.0, 1.0)

    # ── Porte de bruit ───────────────────────────────────────────────────
    processed = _noise_gate(processed, noise_gate_db)

    # ── Normalisation LUFS ───────────────────────────────────────────────
    processed = _normalize_lufs(processed, sr, target_lufs)

    return processed, sr


# ╔══════════════════════════════════════════════════════════╗
# ║  VISUALISATION — WAVEFORM A/B                            ║
# ╚══════════════════════════════════════════════════════════╝

def plot_waveform_ab(original_path: str, processed: np.ndarray, sr: int) -> str:
    """Génère un PNG comparatif waveform avant/après. Retourne le chemin."""
    orig, _ = sf.read(original_path, always_2d=False)
    if orig.dtype != np.float32:
        orig = orig.astype(np.float32)
    if orig.max() > 1.0:
        orig /= 32768.0
    if orig.ndim > 1:
        orig = orig.mean(axis=1)

    proc_mono = processed.mean(axis=1) if processed.ndim > 1 else processed

    t_orig = np.linspace(0, len(orig)/sr, len(orig))
    t_proc = np.linspace(0, len(proc_mono)/sr, len(proc_mono))

    fig, axes = plt.subplots(2, 1, figsize=(12, 5),
                              facecolor='#0d0d0d', sharex=False)
    fig.suptitle('F02 CELESTIAN — Waveform A/B', color='#c0392b',
                 fontfamily='monospace', fontsize=13)

    axes[0].fill_between(t_orig, orig, alpha=0.7, color='#8b0000')
    axes[0].plot(t_orig, orig, color='#c0392b', linewidth=0.4)
    axes[0].set_facecolor('#111')
    axes[0].set_ylabel('AVANT (voix_brute)', color='#888', fontfamily='monospace', fontsize=9)
    axes[0].tick_params(colors='#555')
    axes[0].spines[:].set_color('#333')

    axes[1].fill_between(t_proc, proc_mono, alpha=0.7, color='#1a4a1a')
    axes[1].plot(t_proc, proc_mono, color='#2ecc71', linewidth=0.4)
    axes[1].set_facecolor('#111')
    axes[1].set_ylabel('APRÈS (voix_purifiee)', color='#888', fontfamily='monospace', fontsize=9)
    axes[1].set_xlabel('Temps (s)', color='#555', fontfamily='monospace', fontsize=9)
    axes[1].tick_params(colors='#555')
    axes[1].spines[:].set_color('#333')

    plt.tight_layout()
    out_png = '/tmp/celestian_waveform_ab.png'
    plt.savefig(out_png, dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
    plt.close(fig)
    return out_png


# ╔══════════════════════════════════════════════════════════╗
# ║  HELPERS LIBER + LOG                                     ║
# ╚══════════════════════════════════════════════════════════╝

def _log_f02(event, detail=''):
    log_path = os.path.join(F02_TRACKING, 'F02_LOG.md')
    ts = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(f'\n## [{ts}] {event}\n{detail}\n')


def _update_liber_f02(preset_name, output_path, comp_threshold, reverb_wet, sample_rate=None):
    if not os.path.exists(LIBER_DRIVE):
        return
    with open(LIBER_DRIVE) as f:
        liber = json.load(f)
    liber['f02_celestian']['status']               = 'done'
    liber['f02_celestian']['preset_name']           = preset_name
    liber['f02_celestian']['compression_threshold_db'] = comp_threshold
    liber['f02_celestian']['reverb_wet_level']      = reverb_wet
    liber['f02_celestian']['output_path']           = output_path
    if sample_rate:
        liber['f02_celestian']['sample_rate']       = sample_rate  # X-02
    liber['fleet_status']                           = 'voice_purified_ready'
    liber['sr_custos']['last_validation'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    with open(LIBER_DRIVE, 'w', encoding='utf-8') as f:
        json.dump(liber, f, indent=2, ensure_ascii=False)


def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as fp:
        for chunk in iter(lambda: fp.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()


# ╔══════════════════════════════════════════════════════════╗
# ║  FONCTION PRINCIPALE — purify()                          ║
# ╚══════════════════════════════════════════════════════════╝

def purify(
    input_from_drive,      # bool : utiliser voix_brute.wav du Drive
    input_upload,          # fichier uploadé
    preset_name,
    use_preset_defaults,
    highpass_hz,
    comp_threshold,
    comp_ratio,
    reverb_room,
    reverb_wet,
    presence_boost,
    presence_gain_db,
    noise_gate_db,
    target_lufs,
):
    status_log = []

    # ── Résoudre l'entrée ────────────────────────────────────────────────
    if input_from_drive:
        input_path = os.path.join(F01_OUT, 'voix_brute.wav')
        if not os.path.exists(input_path):
            return None, None, '[ERREUR] voix_brute.wav introuvable dans F01_DOMINION/OUT/. Vérifier F01.'
    elif input_upload is not None:
        input_path = input_upload.name
    else:
        return None, None, '[ERREUR] Aucune entrée sélectionnée.'

    status_log.append(f'[F02] Entrée     : {os.path.basename(input_path)}')
    _dstr = 'oui' if use_preset_defaults else 'non'
    status_log.append(f'[F02] Preset     : {preset_name} (defaults={_dstr})')
    status_log.append(f'[F02] Highpass   : {highpass_hz} Hz')
    status_log.append(f'[F02] Compressor : {comp_threshold} dB | ratio {comp_ratio}:1')
    status_log.append(f'[F02] Reverb     : room {reverb_room} | wet {reverb_wet}')
    status_log.append(f'[F02] Présence   : {presence_boost} (+{presence_gain_db} dB)')
    status_log.append(f'[F02] Gate bruit : {noise_gate_db} dB')
    status_log.append(f'[F02] Cible LUFS : {target_lufs} LUFS')

    try:
        t0 = time.time()
        processed, sr = apply_dsp_pipeline(
            input_path, preset_name,
            highpass_hz=highpass_hz,
            comp_threshold=comp_threshold,
            comp_ratio=comp_ratio,
            reverb_room=reverb_room,
            reverb_wet=reverb_wet,
            presence_boost=presence_boost,
            presence_gain_db=presence_gain_db,
            noise_gate_db=noise_gate_db,
            target_lufs=target_lufs,
            use_preset_defaults=use_preset_defaults,
        )
        elapsed = round(time.time() - t0, 2)
        status_log.append(f'[F02] DSP terminé en {elapsed}s')

        # ── Écriture locale (pour Gradio) ────────────────────────────────
        tmp_out = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        tmp_out.close()
        sf.write(tmp_out.name, processed, sr, subtype='PCM_16')

        # ── Copier vers Drive ────────────────────────────────────────────
        out_drive = os.path.join(F02_OUT, 'voix_purifiee.wav')
        shutil.copy(tmp_out.name, out_drive)
        status_log.append(f'[F02] Déposé Drive : {out_drive}')
        status_log.append(f'[F02] MD5 : {_md5(out_drive)}')

        # ── Waveform A/B ─────────────────────────────────────────────────
        waveform_png = plot_waveform_ab(input_path, processed, sr)
        status_log.append('[F02] Waveform A/B généré.')

        # ── Mise à jour liber ────────────────────────────────────────────
        _update_liber_f02(preset_name, out_drive, comp_threshold, reverb_wet, sr)
        status_log.append('[F02] liber_sanctorum.json → voice_purified_ready')

        _log_f02(
            'PURIFICATION TERMINÉE',
            f'preset={preset_name} | lufs={target_lufs} | elapsed={elapsed}s'
        )

        return tmp_out.name, waveform_png, '\n'.join(status_log)

    except Exception as e:
        import traceback
        status_log.append(f'[ERREUR] {e}')
        status_log.append(traceback.format_exc())
        _log_f02('ERREUR', str(e))
        return None, None, '\n'.join(status_log)


def get_preset_info(preset_name):
    """Retourner la description du preset sélectionné."""
    if preset_name in PRESETS:
        m = PRESETS[preset_name]
        return getattr(m, 'PRESET_DESCRIPTION', '(aucune description)')
    return '(preset inconnu)'


def get_preset_defaults(preset_name):
    """Lire les paramètres par défaut depuis PRESET_DEFAULTS du module preset (A-03)."""
    if preset_name in PRESETS:
        m = PRESETS[preset_name]
        d = getattr(m, 'PRESET_DEFAULTS', None)
        if d:
            return (d.get('highpass_hz', 75.0), d.get('comp_threshold', -15.0),
                    d.get('comp_ratio', 3.5), d.get('reverb_room', 0.20),
                    d.get('reverb_wet', 0.08))
    # Fallback si PRESET_DEFAULTS absent
    fallback = {
        'standard_voix_purifiee': (75.0, -15.0, 3.5, 0.20, 0.08),
        'voix_de_dieu'           : (60.0, -12.0, 4.0, 0.45, 0.20),
        'echo_chamber'           : (80.0, -18.0, 3.0, 0.30, 0.25),
        'micro_aeroport'         : (300., -10.0, 5.0, 0.05, 0.02),
    }
    if preset_name in fallback:
        return fallback[preset_name]
    return 75.0, -15.0, 3.5, 0.20, 0.08


def update_sliders_from_preset(preset_name):
    """Met à jour les sliders quand l'utilisateur change de preset."""
    hp, ct, cr, rr, rw = get_preset_defaults(preset_name)
    desc = get_preset_info(preset_name)
    return hp, ct, cr, rr, rw, desc


# ╔══════════════════════════════════════════════════════════╗
# ║  INTERFACE GRADIO                                        ║
# ╚══════════════════════════════════════════════════════════╝

CSS = """
.gradio-container { background: #0a0a0f; }
#celestian-header { text-align: center; padding: 16px;
    border: 1px solid #1a1a3a;
    background: linear-gradient(135deg, #05051a 0%, #0a0a0f 100%); margin-bottom: 12px; }
#celestian-header h1 { color: #3498db; font-family: monospace; font-size: 1.4em; }
#celestian-header p  { color: #666; font-size: 0.85em; font-family: monospace; }
.label-text { color: #aaa !important; font-family: monospace !important; }
button.primary { background: #1a2a4a !important; color: #aad4f5 !important; }
"""

PRESET_NAMES = list(PRESETS.keys()) or ['standard_voix_purifiee']
DEFAULT_PRESET = 'standard_voix_purifiee' if 'standard_voix_purifiee' in PRESET_NAMES else PRESET_NAMES[0]

with gr.Blocks(css=CSS, title='F02 CELESTIAN — La Montre') as demo:

    gr.HTML("""
    <div id='celestian-header'>
      <h1>F02 — CELESTIAN : La Montre</h1>
      <p>Frégate SANCTORUM · Purification DSP · Spotify Pedalboard</p>
      <p>Ordre de la Rose Sacrée — Adepta Sororitas</p>
    </div>
    """)

    with gr.Tabs():

        # ── Onglet 1 : Purification ────────────────────────────────────
        with gr.Tab('Purification DSP'):

            with gr.Row():
                with gr.Column(scale=2):
                    gr.Markdown('### Entrée audio')
                    input_from_drive = gr.Checkbox(
                        label='Utiliser voix_brute.wav depuis Drive (F01_DOMINION/OUT/)',
                        value=True
                    )
                    input_upload = gr.File(
                        label='Ou uploader un fichier audio',
                        file_types=['.wav', '.mp3', '.flac', '.ogg']
                    )

                with gr.Column(scale=3):
                    gr.Markdown('### Preset personnage')
                    preset_selector = gr.Dropdown(
                        label='Preset',
                        choices=PRESET_NAMES,
                        value=DEFAULT_PRESET
                    )
                    preset_desc = gr.Textbox(
                        label='Description du preset',
                        value=get_preset_info(DEFAULT_PRESET),
                        interactive=False,
                        lines=1
                    )
                    use_preset_defaults = gr.Checkbox(
                        label='Utiliser les valeurs par défaut du preset (décocher pour override manuel)',
                        value=True
                    )

            gr.Markdown('### Paramètres DSP (actifs si override manuel activé)')
            with gr.Row():
                highpass_hz = gr.Slider(
                    label='Highpass (Hz)', minimum=20, maximum=500, step=5, value=75
                )
                comp_threshold = gr.Slider(
                    label='Compressor Threshold (dB)', minimum=-40, maximum=0, step=1, value=-15
                )
                comp_ratio = gr.Slider(
                    label='Compressor Ratio', minimum=1.0, maximum=10.0, step=0.5, value=3.5
                )

            with gr.Row():
                reverb_room = gr.Slider(
                    label='Reverb Room Size', minimum=0.0, maximum=1.0, step=0.01, value=0.20
                )
                reverb_wet = gr.Slider(
                    label='Reverb Wet Level', minimum=0.0, maximum=0.5, step=0.01, value=0.08
                )

            gr.Markdown('### Options avancées')
            with gr.Row():
                presence_boost = gr.Checkbox(
                    label='Boost de présence (2–5 kHz)', value=False
                )
                presence_gain_db = gr.Slider(
                    label='Gain présence (dB)', minimum=0, maximum=12, step=0.5, value=3.0
                )
                noise_gate_db = gr.Slider(
                    label='Porte de bruit (dB, -100 = désactivé)',
                    minimum=-100, maximum=-20, step=5, value=-50
                )
                target_lufs = gr.Slider(
                    label='Cible LUFS (normalisation)', minimum=-24, maximum=-8, step=1, value=-16
                )

            purify_btn = gr.Button('PURIFIER — CELESTIAN', variant='primary', size='lg')

            with gr.Row():
                audio_out = gr.Audio(
                    label='voix_purifiee.wav (F02_CELESTIAN/OUT/)',
                    type='filepath', scale=3
                )
                status_out = gr.Textbox(
                    label='Journal de mission',
                    lines=14, interactive=False, scale=2
                )

        # ── Onglet 2 : Waveform A/B ────────────────────────────────────
        with gr.Tab('Waveform A/B'):
            gr.Markdown('### Comparaison avant/après purification')
            waveform_img = gr.Image(
                label='Waveform — voix_brute vs voix_purifiee',
                type='filepath'
            )
            gr.Markdown(
                '*Le graphique est généré automatiquement après chaque purification.*'
            )

        # ── Onglet 3 : Presets Manager ─────────────────────────────────
        with gr.Tab('Presets Chargés'):
            gr.Markdown('### Presets dynamiquement chargés depuis `F02_CELESTIAN/CODEBASE/presets/`')
            presets_display = gr.JSON(
                label='Presets disponibles',
                value={
                    name: getattr(PRESETS[name], 'PRESET_DESCRIPTION', '?')
                    for name in PRESETS
                }
            )
            gr.Markdown(
                '> Pour ajouter un nouveau personnage : déposer un `.py` dans '
                '`SANCTORUM/F02_CELESTIAN/CODEBASE/presets/` puis relancer la cellule.'
            )

        # ── Onglet 4 : Statut Flotte ───────────────────────────────────
        with gr.Tab('Statut Flotte'):
            gr.Markdown('### liber_sanctorum.json')
            liber_out = gr.JSON(label='État de la flotte')
            read_liber_btn = gr.Button('Lire le Liber')

    # ── Câblage ──────────────────────────────────────────────────────────

    # Sync sliders au changement de preset
    preset_selector.change(
        update_sliders_from_preset,
        inputs=[preset_selector],
        outputs=[highpass_hz, comp_threshold, comp_ratio, reverb_room, reverb_wet, preset_desc]
    )

    # Purification
    purify_btn.click(
        purify,
        inputs=[
            input_from_drive, input_upload,
            preset_selector, use_preset_defaults,
            highpass_hz, comp_threshold, comp_ratio,
            reverb_room, reverb_wet,
            presence_boost, presence_gain_db,
            noise_gate_db, target_lufs,
        ],
        outputs=[audio_out, waveform_img, status_out]
    )

    read_liber_btn.click(
        lambda: json.load(open(LIBER_DRIVE)) if os.path.exists(LIBER_DRIVE) else {'error': 'liber non trouvé'},
        inputs=[],
        outputs=[liber_out]
    )

# ── Lancer ──────────────────────────────────────────────────────────────
print('[GRADIO] Démarrage interface La Montre...')
demo.launch(share=True, debug=True, server_port=7861, inbrowser=False)

---
## CELLULE 4 — SR_CUSTOS CHECK-IN

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — SR_CUSTOS CHECK-IN F02                     ║
# ╚══════════════════════════════════════════════════════════╝

import os, sys, json, shutil

DRIVE_ROOT    = '/content/drive/MyDrive/SANCTORUM'
SANCTORUM_DIR = '/content/SANCTORUM'
LIBER_DRIVE   = f'{DRIVE_ROOT}/liber_sanctorum.json'
OUT_PATH      = f'{DRIVE_ROOT}/F02_CELESTIAN/OUT/voix_purifiee.wav'

if not os.path.exists(OUT_PATH):
    print('[SR_CUSTOS] ERREUR: voix_purifiee.wav absent. Lancer la cellule 3 d\'abord.')
else:
    custos = os.path.join(SANCTORUM_DIR, 'SR_CUSTOS.py')
    if os.path.exists(custos):
        local_liber = os.path.join(SANCTORUM_DIR, 'liber_sanctorum.json')
        shutil.copy(LIBER_DRIVE, local_liber)
        !python {custos} --mode check-in --frigate F02 --output {OUT_PATH}
        shutil.copy(local_liber, LIBER_DRIVE)

# ── Sync logs CUSTOS vers Drive (M-04) ─────────────────────────────────
for _log_name in ['SR_CAMPAIGN_LOG.md', 'SR_TRANSFER_LOG.md']:
    _log_src = os.path.join(SANCTORUM_DIR, 'TRACKING', _log_name)
    if os.path.exists(_log_src):
        shutil.copy(_log_src, os.path.join(DRIVE_ROOT, 'TRACKING', _log_name))
    else:
        print('[SR_CUSTOS] SR_CUSTOS.py absent — mise à jour manuelle.')
        with open(LIBER_DRIVE) as f:
            liber = json.load(f)
        liber['f02_celestian']['status']   = 'done'
        liber['f02_celestian']['output_path'] = OUT_PATH
        liber['fleet_status'] = 'voice_purified_ready'
        with open(LIBER_DRIVE, 'w') as f:
            json.dump(liber, f, indent=2, ensure_ascii=False)

    with open(LIBER_DRIVE) as f:
        liber = json.load(f)
    print('\n╔══════════════════════════════════════════════════════╗')
    print('║          ÉTAT DE LA FLOTTE — POST F02                 ║')
    print('╠══════════════════════════════════════════════════════╣')
    print(f"║  fleet_status : {liber.get('fleet_status','n/a'):<36}║")
    print(f"║  F01 DOMINION : {liber['f01_dominion']['status']:<36}║")
    print(f"║  F02 CELESTIAN: {liber['f02_celestian']['status']:<36}║")
    print(f"║  F03 SERAPHIM : {liber['f03_seraphim']['status']:<36}║")
    print('╠══════════════════════════════════════════════════════╣')
    print('║  PROCHAINE ÉTAPE : F03_SERAPHIM (Mix Audio Final)    ║')
    print('╚══════════════════════════════════════════════════════╝')